# YOLO26n + Improved Backbone (ADown + C3k2_DCNv2): Authorized/Unauthorized Detection

**Model:** Improved backbone + original YOLO26n neck + head

| Part | Change |
|---|---|
| Backbone | P3/P4/P5 down-sampling `Conv` → **ADown**; 3×3 conv inside the P4 and P5 C3k2 stages → **DCNv2** |
| Neck | Same as YOLO26n (unchanged) |
| Head | Same as YOLO26n (unchanged) |

**Kaggle settings:** Accelerator = **GPU T4 x1** (single GPU, no DDP), Internet = **On**

**Steps:** 1) setup → 2) dataset → 3) module → 4) model check → 5) 4 ablation trainings → 6) test-set comparison → 7) unauthorized threshold + FAR → 8) NCNN export for Pi 5

## 1. Setup

In [ ]:
# ultralytics version is pinned: this code was tested with 8.4.137
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, glob, json, shutil, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so dcn_backbone.py can be imported

## 2. Dataset (download from Roboflow)

**API key:** In Kaggle, add your key under **Add-ons → Secrets** with the name `ROBOFLOW_API_KEY`. If there is no secret, put the key in the `API_KEY` variable below.
Do not write the key directly in the code if the notebook will be public.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# API key: read from Kaggle Secrets; if not available, write it here
API_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception:
    pass
assert API_KEY, "Add the ROBOFLOW_API_KEY secret or set API_KEY"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: steps 6 and 7 need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

# 3. How to check this yourself in Kaggle

In [ ]:
from PIL import Image
from collections import Counter

for split in ["train", "valid", "test"]:
    sizes = Counter(
        Image.open(p).size  # (width, height)
        for p in (DATA_ROOT / split / "images").glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    )
    total = sum(sizes.values())
    print(f"\n{split}: {total} images")
    for (w, h), n in sizes.most_common():
        print(f"  {w} x {h}: {n} ({n / total:.1%})")


## 3. Custom module: DCNv2 + DCNTrainer

- `DCNv2`: modulated deformable conv (`torchvision.ops.DeformConv2d`). The offset/mask conv is zero-initialised, so training starts out behaving like a plain conv.
- `add_dcnv2`: replaces the 3×3 conv in the C3k2 bottlenecks of backbone layers 6 (P4) and 8 (P5).
- `DCNTrainer`: builds the model, inserts DCN, **then** loads the pretrained `yolo26n.pt` weights. All matching layers stay pretrained; only the DCN layers learn from scratch.
- No changes to the ultralytics source code are needed.

In [ ]:
%%writefile dcn_backbone.py
"""DCNv2 backbone patch for YOLO26n: replaces the 3x3 conv inside the C3k2 bottlenecks at P4/P5 with DCNv2."""
import torch
import torch.nn as nn
from torchvision.ops import DeformConv2d

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import Bottleneck
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK


class DCNv2(nn.Module):
    """Modulated deformable conv (DCNv2) + BN + SiLU, drop-in replacement for ultralytics Conv(k=3, s=1)."""

    def __init__(self, c1, c2, k=3, s=1, p=None, g=1, d=1):
        super().__init__()
        p = k // 2 if p is None else p
        # 2*k*k offsets + k*k modulation mask, zero-init so training starts as a plain conv
        self.conv_offset_mask = nn.Conv2d(c1, 3 * k * k, k, s, p, dilation=d, bias=True)
        nn.init.zeros_(self.conv_offset_mask.weight)
        nn.init.zeros_(self.conv_offset_mask.bias)
        self.dcn = DeformConv2d(c1, c2, k, s, p, dilation=d, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()

    def forward(self, x):
        o1, o2, mask = torch.chunk(self.conv_offset_mask(x), 3, dim=1)
        offset = torch.cat((o1, o2), dim=1)
        if not torch.jit.is_tracing():
            # guard against NaN/huge offsets (e.g. the uninitialised FLOPs-probe input), which crash the CPU kernel
            lim = float(max(x.shape[-2:]))
            offset = torch.nan_to_num(offset, 0.0, lim, -lim).clamp(-lim, lim)
        return self.act(self.bn(self.dcn(x, offset, torch.sigmoid(mask))))


def add_dcnv2(model, layers=(6, 8)):
    """Swap the 3x3 cv2 conv of every Bottleneck inside the given backbone layers for DCNv2."""
    n = 0
    for i in layers:
        for m in model.model[i].modules():
            if isinstance(m, Bottleneck) and isinstance(m.cv2, Conv) and m.cv2.conv.kernel_size == (3, 3):
                c = m.cv2.conv
                m.cv2 = DCNv2(c.in_channels, c.out_channels, 3, c.stride[0], g=c.groups)
                n += 1
    return n


class DCNTrainer(DetectionTrainer):
    """DetectionTrainer that inserts DCNv2 before loading pretrained weights (DCN layers start from scratch)."""

    dcn_layers = (6, 8)

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        n = add_dcnv2(model, self.dcn_layers)
        if RANK in {-1, 0}:
            print(f"DCNv2: replaced {n} conv(s) in layers {self.dcn_layers}")
        if weights:
            model.load(weights)
        if verbose and RANK in {-1, 0}:
            model.info()
        return model

## 4. Improved backbone yaml (ADown) + model check

In [ ]:
%%writefile yolo26n-adown.yaml
# YOLO26n + ADown backbone (neck + head = YOLO26n original)
nc: 27
end2end: True
reg_max: 1
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, ADown, [256]] # 3-P3/8   (Conv -> ADown)
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, ADown, [512]] # 5-P4/16  (Conv -> ADown)
  - [-1, 2, C3k2, [512, True]] # 6   (DCNv2 inserted by DCNTrainer)
  - [-1, 1, ADown, [1024]] # 7-P5/32 (Conv -> ADown)
  - [-1, 2, C3k2, [1024, True]] # 8  (DCNv2 inserted by DCNTrainer)
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

# YOLO26n neck + head (unchanged)
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16 (P3/8)
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, True]] # 19 (P4/16)
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 22 (P5/32)
  - [[16, 19, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)

In [ ]:
import importlib, dcn_backbone
importlib.reload(dcn_backbone)
from dcn_backbone import add_dcnv2, DCNTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

BASE_YAML = "yolo26n.yaml"
ADOWN_YAML = str(WORK / "yolo26n-adown.yaml")

# 4 ablation configurations: (name, yaml, uses DCN)
VARIANTS = [
    ("1_base",       BASE_YAML,  False),
    ("2_adown",      ADOWN_YAML, False),
    ("3_dcnv2",      BASE_YAML,  True),
    ("4_adown_dcnv2", ADOWN_YAML, True),   # your improved model
]

rows = []
for name, cfg, dcn in VARIANTS:
    m = DetectionModel(yaml_model_load(cfg), nc=len(names), verbose=False)
    n_dcn = add_dcnv2(m) if dcn else 0
    x = torch.zeros(1, 3, 640, 640)
    m.eval(); _ = m(x)  # forward check
    rows.append({"model": name, "DCN layers": n_dcn,
                 "Params (M)": round(sum(p.numel() for p in m.parameters()) / 1e6, 3),
                 "GFLOPs*": round(get_flops(m, 640), 2)})
pd.DataFrame(rows)

> \* thop does not count the FLOPs of `DeformConv2d`, so the DCN models show slightly lower GFLOPs than they really have. Mention this in a footnote in the paper, or use FPS as the main speed metric.

## 5. Training (4 ablations)

All models use **the same settings** and start from the same pretrained `yolo26n.pt`, so the comparison is fair.
On a T4, each model should take roughly 1–1.5 hours for 100 epochs. A Kaggle session is limited to 12 hours, so run each cell in a separate session if needed.

In [ ]:
EPOCHS = 100
IMGSZ  = 640
BATCH  = 16         # 32 can run out of memory on Kaggle; keep the same value for all 4 models
WORKERS = 2         # fewer dataloader processes = less CPU RAM
SEED   = 0          # for the paper, run seeds 0, 1, 2 and report mean ± std
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # Fixed optimizer (what YOLO26 picks for long runs), same for all 4 models and every EPOCHS value.
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

import gc
from ultralytics import YOLO

def show_memory(tag=""):
    """Print free CPU RAM and GPU memory (to see what is filling up)."""
    import psutil
    ram = psutil.virtual_memory()
    msg = f"[{tag}] RAM used {ram.used / 1e9:.1f}/{ram.total / 1e9:.1f} GB"
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        msg += f" | GPU used {(total - free) / 1e9:.1f}/{total / 1e9:.1f} GB"
    print(msg)

def train_variant(name, cfg, dcn, seed=SEED):
    run_name = f"{name}_s{seed}"
    show_memory(f"before {run_name}")
    model = YOLO(cfg)
    kw = dict(COMMON, seed=seed, name=run_name)
    if dcn:
        model.train(trainer=DCNTrainer, **kw)
    else:
        model.train(**kw)
    # free memory before the next model is trained in the same session
    del model
    gc.collect()
    torch.cuda.empty_cache()
    show_memory(f"after {run_name}")
    return Path(PROJECT) / run_name / "weights" / "best.pt"

BEST = {}
show_memory("start")

In [ ]:
# 1) Baseline YOLO26n
BEST["1_base"] = train_variant(*VARIANTS[0])

In [ ]:
# 2) + ADown
BEST["2_adown"] = train_variant(*VARIANTS[1])

In [ ]:
# 3) + C3k2_DCNv2
BEST["3_dcnv2"] = train_variant(*VARIANTS[2])

In [ ]:
# 4) + ADown + C3k2_DCNv2  (Improved model)
BEST["4_adown_dcnv2"] = train_variant(*VARIANTS[3])

## 6. Test-set comparison (ablation table)

In [ ]:
import time
import dcn_backbone  # must be imported before loading a DCN checkpoint

# after a session restart, find best.pt from earlier runs
for name, _, _ in VARIANTS:
    p = Path(PROJECT) / f"{name}_s{SEED}" / "weights" / "best.pt"
    if p.exists():
        BEST[name] = p

def gpu_latency_ms(model, n=200):
    """batch=1 GPU latency (ms, fp32), model forward only"""
    net = model.model.cuda().eval().float()
    x = torch.zeros(1, 3, IMGSZ, IMGSZ, device="cuda")
    with torch.no_grad():
        for _ in range(30): net(x)
        torch.cuda.synchronize(); t = time.perf_counter()
        for _ in range(n): net(x)
        torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000

rows = []
for name, w in BEST.items():
    m = YOLO(str(w))
    r = m.val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    rows.append({
        "Model": name,
        "P": round(r.box.mp, 4), "R": round(r.box.mr, 4),
        "mAP50": round(r.box.map50, 4), "mAP50-95": round(r.box.map, 4),
        "Params (M)": round(sum(p.numel() for p in m.model.parameters()) / 1e6, 3),
        "GPU ms (T4, fp32)": round(gpu_latency_ms(YOLO(str(w))), 2),
    })
results = pd.DataFrame(rows)
results.to_csv(WORK / "ablation_test_results.csv", index=False)
results

## 7. Authorized / Unauthorized: threshold τ and FAR

The model only knows the 27 enrolled people. **Rule:** if the highest confidence is < τ, the person is treated as **Unauthorized**.

- τ is chosen on the **validation** set so that 95% of correctly recognised authorized people are still accepted.
- Measuring **FAR (False Accept Rate)** needs **images of unknown people**, i.e. anyone outside the 27. Add them as a separate Kaggle dataset and set `UNKNOWN_DIR`. Without them, only the authorized results are shown.

In [ ]:
UNKNOWN_DIR = None   # example: "/kaggle/input/unknown-people/images"
TARGET_TAR = 0.95    # pick τ so that 95% of correct authorized detections are accepted

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

def top1_records(model, split):
    """Top-confidence detection per image: (conf, is_correct)"""
    recs = []
    img_dir = DATA_ROOT / split / "images"
    for r in model.predict(str(img_dir), imgsz=IMGSZ, conf=0.001, device=0, stream=True, verbose=False):
        lbl = DATA_ROOT / split / "labels" / (Path(r.path).stem + ".txt")
        gts = [list(map(float, l.split())) for l in open(lbl)] if lbl.exists() else []
        if not gts:
            continue
        if len(r.boxes) == 0:
            recs.append((0.0, False)); continue
        i = int(r.boxes.conf.argmax())
        conf, cls = float(r.boxes.conf[i]), int(r.boxes.cls[i])
        box = r.boxes.xyxyn[i].tolist()
        ok = any(int(g[0]) == cls and iou(box, [g[1]-g[3]/2, g[2]-g[4]/2, g[1]+g[3]/2, g[2]+g[4]/2]) >= 0.5 for g in gts)
        recs.append((conf, ok))
    return recs

def unknown_top_conf(model, folder):
    return [float(r.boxes.conf.max()) if len(r.boxes) else 0.0
            for r in model.predict(folder, imgsz=IMGSZ, conf=0.001, device=0, stream=True, verbose=False)]

rows = []
for name, w in BEST.items():
    m = YOLO(str(w))
    val = top1_records(m, "valid")
    correct_conf = np.array([c for c, ok in val if ok])
    tau = float(np.quantile(correct_conf, 1 - TARGET_TAR)) if len(correct_conf) else 0.5

    test = top1_records(m, "test")
    n = len(test)
    accepted_ok  = sum(1 for c, ok in test if ok and c >= tau)        # correctly identified and accepted
    accepted_bad = sum(1 for c, ok in test if (not ok) and c >= tau)  # accepted under the wrong identity (misidentification)
    row = {"Model": name, "tau": round(tau, 3),
           "Authorized accept (TAR)": round(accepted_ok / n, 4),
           "Misidentified": round(accepted_bad / n, 4),
           "Rejected (false alarm)": round(1 - (accepted_ok + accepted_bad) / n, 4)}
    if UNKNOWN_DIR:
        u = np.array(unknown_top_conf(m, UNKNOWN_DIR))
        row["Unknown FAR"] = round(float((u >= tau).mean()), 4)
        row["#unknown imgs"] = len(u)
    rows.append(row)

auth = pd.DataFrame(rows)
auth.to_csv(WORK / "authorized_unauthorized_results.csv", index=False)
auth

## 8. NCNN export for Raspberry Pi 5

DCNv2 exports to NCNN's native `DeformableConv2D` layer (tested locally). **ONNX export does not work**, because ONNX does not support `deform_conv2d`. Use **NCNN** on the Pi 5.

In [ ]:
PI_IMGSZ = 320   # test both 320 and 416 on the Pi 5; the boxes are large, so a small input will likely still work well

export_dir = WORK / "pi5_models"
export_dir.mkdir(exist_ok=True)
for name, w in BEST.items():
    p = YOLO(str(w)).export(format="ncnn", imgsz=PI_IMGSZ, half=False)
    dst = export_dir / f"{name}_ncnn_{PI_IMGSZ}"
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(p, dst)
    n_dcn = open(dst / "model.ncnn.param").read().count("DeformableConv2D")
    print(f"{name:16s} -> {dst.name}  (DeformableConv2D layers: {n_dcn})")

In [ ]:
%%writefile pi5_models/pi5_benchmark.py
# Run on Raspberry Pi 5:  pip install ultralytics ncnn  &&  python pi5_benchmark.py
import glob, time
import numpy as np
from ultralytics import YOLO

IMGSZ = 320
for d in sorted(glob.glob(f"*_ncnn_{IMGSZ}")):
    m = YOLO(d, task="detect")
    img = (np.random.rand(IMGSZ, IMGSZ, 3) * 255).astype("uint8")
    for _ in range(10):
        m.predict(img, imgsz=IMGSZ, verbose=False)
    t, n = time.perf_counter(), 100
    for _ in range(n):
        m.predict(img, imgsz=IMGSZ, verbose=False)
    ms = (time.perf_counter() - t) / n * 1000
    print(f"{d:30s} {ms:7.1f} ms   {1000 / ms:5.1f} FPS")

## 9. Zip all results for download

In [ ]:
import zipfile
shutil.copy("dcn_backbone.py", WORK / "pi5_models" / "dcn_backbone.py")
shutil.copy("yolo26n-adown.yaml", WORK / "pi5_models" / "yolo26n-adown.yaml")
keep = ["runs", "pi5_models", "ablation_test_results.csv", "authorized_unauthorized_results.csv",
        "dcn_backbone.py", "yolo26n-adown.yaml", "data.yaml"]
with zipfile.ZipFile(WORK / "paper1_backbone_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k in keep:
        p = WORK / k
        for f in ([p] if p.is_file() else p.rglob("*") if p.exists() else []):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
print("Download: /kaggle/working/paper1_backbone_results.zip  (from the Output tab)")